In [ ]:
tables = spark.sql("SHOW TABLES IN silver").collect()

matches = []

for row in tables:
    table_name = row["tableName"]
    full_name = f"silver.{table_name}"
    try:
        cols = spark.sql(f"DESCRIBE {full_name}").collect()
        for c in cols:
            col_name = c["col_name"]
            if col_name:
                cname = col_name.strip().lower()
                if "cprod" in cname or "group_conformed" in cname:
                    matches.append((full_name, col_name))
    except Exception:
        pass

display(spark.createDataFrame(matches, ["table_name", "column_name"]))

entity ------> care Episode service_src_id

MPB-------------

In [ ]:
--code 1
rserv.service_id as care_epi_service_id

In [ ]:
--code 2
LEFT JOIN
    silver_rdm_service rserv
        ON rserv.service_src_sys_inst_id = 'MPB001'
       AND trim(lower(rserv.service_src_name)) = trim(lower(ten.client_type))
       AND rserv.z_order_is_active = 1

WIP--------


In [ ]:
-- Added as per Monday definition: map WIP service to RDM service_id

, rserv.service_id as care_epi_service_id

In [ ]:
-- Map WIP service type description to RDM service source name
LEFT JOIN
    silver_rdm_service rserv
        ON rserv.service_src_sys_inst_id = 'WIP001'
       AND trim(lower(rserv.service_src_name)) = trim(lower(servt.description))
       AND rserv.z_order_is_active = 1

In [ ]:
s1--------------------------

In [ ]:
, rserv.service_id as care_epi_service_id

In [ ]:
LEFT JOIN
    silver_rdm_service rserv
        ON rserv.service_src_sys_inst_id = CONCAT('SONE', r.id_organisation_source)
       AND trim(lower(rserv.service_src_name)) = trim(lower(serv.configured_list_option))
       AND rserv.z_order_is_active = 1

In [ ]:
%%sql
SELECT
    care_epi_service_id,
    COUNT(*) as cnt
FROM silver_care_episode
WHERE z_src_system_id = 'SONE'
GROUP BY care_epi_service_id
ORDER BY cnt DESC;

In [ ]:
%%sql
SELECT TOP 100
    care_epi_id,
    care_epi_service_id,
    z_src_system_id,
    z_src_system_instance
FROM silver_care_episode
WHERE z_src_system_id = 'SONE';

In [ ]:
Implemented care_epi_service_id for SONE in Care Episode Silver Table.
Mapped SystemOne service value from serv.configured_list_option to silver_rdm_service.service_src_name using source system instance CONCAT('SONE', r.id_organisation_source), and populated care_epi_service_id from silver_rdm_service.service_id.

Validation completed:

Confirmed successful population where SONE org-source-specific RDM mappings exist
Sample validated matches included:
Advice/consultation -> 511
Rheumatology -> 509
Some rows remain null where corresponding SONE RDM mappings are not currently available

Status: Ready for UAT

In [ ]:
The source only provides the service value in serv.configured_list_option, while the Monday definition expects the RDM Service key. So we matched the SONE service value to silver_rdm_service.service_src_name within the correct source system instance using CONCAT('SONE', r.id_organisation_source), and then populated care_epi_service_id from silver_rdm_service.service_id. This ensures the value comes from the correct org-source-specific RDM mapping rather than from a raw text field.